In [ ]:
!pip install -q transformers datasets sacrebleu sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.0 MB/s eta 0:00:00


In [ ]:
# !ls /kaggle/input

# # Example FLORES path
# base_path = "/kaggle/input/datasets/mathurinache/flores101/flores101_dataset"

import kagglehub

# Download latest version
base_path = kagglehub.dataset_download("mathurinache/flores101")

print("FLORES dataset ready.")
print("Base path:", base_path)

datasets
FLORES dataset ready.
Base path: /kaggle/input/datasets/mathurinache/flores101/flores101_dataset


In [ ]:
import os

dev_path = f"{base_path}/devtest"

def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f]

SRC = "eng"
TGT = "zul"

sources = load_lines(f"{dev_path}/{SRC}.devtest")
references = load_lines(f"{dev_path}/{TGT}.devtest")

print("Eval samples:", len(sources))

Eval samples: 1012


In [ ]:
from datasets import load_dataset

# Chosen corpus: OPUS100
dataset = load_dataset("opus100", "en-zu")

train_data = dataset["train"].select(range(20000))  # small subset for Colab

def preprocess(example):
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["zu"]
    }

train_data = train_data.map(preprocess)

README.md: 0.00B [00:00, ?B/s]

en-zu/test-00000-of-00001.parquet:   0%|          | 0.00/58.1k [00:00<?, ?B/s]

en-zu/train-00000-of-00001.parquet:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

en-zu/validation-00000-of-00001.parquet:   0%|          | 0.00/58.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/38616 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "AfriNLP/AfriNLLB-12enc-12dec-full-ft-kd"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
def tokenize(example):
    model_inputs = tokenizer(
        example["src"],
        text_target=example["tgt"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    return model_inputs

tokenized_train = train_data.map(tokenize, batched=True)

print(tokenized_train[0].keys())

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

dict_keys(['translation', 'src', 'tgt', 'input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./afrinllb-retrained",
    per_device_train_batch_size=8,
    num_train_epochs=2,  # keep small for Colab
    learning_rate=2e-5,
    logging_steps=100,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,0.614528
200,0.436220
300,0.375228
400,0.345192
500,0.351310
600,0.346972
700,0.299726
800,0.300972
900,0.288767
1000,0.300059


TrainOutput(global_step=2500, training_loss=0.29778854904174806, metrics={'train_runtime': 5014.6072, 'train_samples_per_second': 7.977, 'train_steps_per_second': 0.499, 'total_flos': 1.083552301056e+16, 'train_loss': 0.29778854904174806, 'epoch': 2.0})

In [ ]:
model.save_pretrained("afrinllb-retrained")
tokenizer.save_pretrained("afrinllb-retrained")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('afrinllb-retrained/tokenizer_config.json',
 'afrinllb-retrained/tokenizer.json')

In [ ]:
import time

def translate(texts, batch_size=8):
    tokenizer.src_lang = "eng_Latn"
    outputs = []

    start = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("zul_Latn"),
                max_length=200
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(decoded)

    total = time.time() - start
    latency = total / len(texts)
    throughput = len(texts) / total

    return outputs, latency, throughput

In [ ]:
predictions, latency, throughput = translate(sources)

print("Latency:", latency)
print("Throughput:", throughput)

Latency: 0.16947557591638074
Throughput: 5.900555254601406


In [ ]:
import sacrebleu

chrf = sacrebleu.corpus_chrf(predictions, [references])
print("chrF++:", chrf.score)

chrF++: 54.773591766630815


In [ ]:
import json

# =========================================================
# Save retrained model metrics
# =========================================================

results = {
    "model": "AfriNLLB-retrained",
    "source_lang": "eng_Latn",
    "target_lang": "zul_Latn",
    "evaluation_dataset": "FLORES-original",
    "training_data": "OPUS100",
    "chrF++": chrf.score,
    "latency": latency,
    "throughput": throughput,
    "num_samples": len(sources)
}

# Save metrics
with open("retrained_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

# =========================================================
# Save retrained model predictions
# =========================================================

with open("retrained_predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)

# =========================================================
# Save evaluation data for separate AfriCOMET notebook
# =========================================================

with open("retrained_sources.json", "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=2)

with open("retrained_references.json", "w", encoding="utf-8") as f:
    json.dump(references, f, indent=2)

print("Saved retrained evaluation files.")

Saved retrained evaluation files.
